In [1]:
"""
mv_plot.py — Figures for Chapter 6 of the thesis
==================================================
Loads results from mv_RL.py (file mv_results.npz) and produces
the thesis figures in PDF and PNG.

Usage:
  1. Run mv_RL.py first  (produces mv_results.npz)
  2. Then run this script:
       python mv_plot.py

Figures produced:
  figure1_convergence_real.pdf/png  — Convergence of phi_1
  figure2_wealth_real.pdf/png       — Terminal wealth distribution (bootstrap OOS)
  figure3_policy_real.pdf/png       — Shape of the learned policy
  convergenza_RL.pdf/png            — Training convergence (real data only)
  rl_out_of_sample.pdf/png          — Chronological OOS wealth 2020-2023
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import os

# ─────────────────────────────────────────────────────────────────────────────
# Global style
# ─────────────────────────────────────────────────────────────────────────────
mpl.rcParams.update({
    'font.family':      'serif',
    'font.size':        11,
    'axes.titlesize':   12,
    'axes.labelsize':   12,
    'legend.fontsize':  9,
    'figure.dpi':       150,
})

COLORS = {
    'real': 'steelblue',
    'bs':   'darkorange',
    'mjd':  'forestgreen',
    'bh':   'gray',
}

RESULTS_FILE = 'mv_results.npz'
OUTPUT_DIR   = '.'

outdir = os.path.expanduser(OUTPUT_DIR)
os.makedirs(outdir, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# Load results
# ─────────────────────────────────────────────────────────────────────────────
print(f"Loading results from: {RESULTS_FILE}")
if not os.path.exists(RESULTS_FILE):
    raise FileNotFoundError(
        f"File {RESULTS_FILE} not found.\n"
        "Run mv_RL.py first to generate it."
    )

data     = np.load(RESULTS_FILE)
h_real   = data['h_real']
h_bs     = data['h_bs']
h_mjd    = data['h_mjd']
tw_real  = data['tw_real']
tw_bs    = data['tw_bs']
tw_mjd   = data['tw_mjd']
phi_real = data['phi_real']
phi_bs   = data['phi_bs']
phi_mjd  = data['phi_mjd']
PHI1_BS  = float(data['PHI1_BS'])
PHI1_MJD = float(data['PHI1_MJD'])

# Chronological OOS wealth paths (2020-2023)
oos_rl  = data['oos_wealth_rl']   if 'oos_wealth_rl'  in data else None
oos_bh  = data['oos_wealth_bh']   if 'oos_wealth_bh'  in data else None
oos_bs  = data['oos_wealth_bs']   if 'oos_wealth_bs'  in data else None
oos_mjd = data['oos_wealth_mjd']  if 'oos_wealth_mjd' in data else None

N_ITER = len(h_real)
Z      = 1.4

print(f"  N_ITER = {N_ITER}")
print(f"  Final phi_1:  Real={phi_real[0]:.4f}  BS={phi_bs[0]:.4f}  MJD={phi_mjd[0]:.4f}")
print(f"  Analytical phi_1*:  BS={PHI1_BS:.4f}  MJD={PHI1_MJD:.4f}")
print(f"  Output folder: {outdir}")

# ─────────────────────────────────────────────────────────────────────────────
# Utilities
# ─────────────────────────────────────────────────────────────────────────────

def smooth(x, w=300):
    if w >= len(x):
        w = max(1, len(x)//10)
    return np.convolve(x, np.ones(w)/w, mode='valid')

def savefig(name):
    for ext in ['pdf', 'png']:
        path = os.path.join(outdir, f"{name}.{ext}")
        plt.savefig(path, dpi=150, bbox_inches='tight')
        print(f"  Saved: {path}")
    plt.close()


# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 1 — Convergence of phi_1 (all three simulators)
# ─────────────────────────────────────────────────────────────────────────────
print("\n[Figure 1] Convergence of phi_1...")

iters = np.arange(1, N_ITER+1)
w     = 300

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: all three curves
ax = axes[0]
for h, lbl, col, ls in [
    (h_real, 'Real S&P 500 data', COLORS['real'], '-'),
    (h_bs,   'BS simulator',      COLORS['bs'],   '--'),
    (h_mjd,  'MJD simulator',     COLORS['mjd'],  '-.'),
]:
    ax.plot(iters[w-1:], smooth(h, w), color=col, lw=2.0, ls=ls, label=lbl)

ax.axhline(PHI1_BS,  color=COLORS['bs'],  lw=1.2, ls=':',
           label=fr'$\phi_1^*$ BS = {PHI1_BS:.4f}')
ax.axhline(PHI1_MJD, color=COLORS['mjd'], lw=1.2, ls=':',
           label=fr'$\phi_1^*$ MJD = {PHI1_MJD:.4f}')
ax.set_xlabel('Iteration')
ax.set_ylabel(r'$\phi_1$', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)

# Right panel: zoom on second half (real data only)
ax = axes[1]
zoom = N_ITER // 2
ax.plot(iters[zoom:],       h_real[zoom:],
        color=COLORS['real'], lw=0.5, alpha=0.3)
ax.plot(iters[zoom+w-1:],  smooth(h_real[zoom:], w),
        color=COLORS['real'], lw=2.0, label='Real data (smoothed)')
ax.axhline(PHI1_BS,  color=COLORS['bs'],  lw=1.5, ls='--',
           label=fr'$\phi_1^*$ BS = {PHI1_BS:.4f}')
ax.axhline(PHI1_MJD, color=COLORS['mjd'], lw=1.5, ls='-.',
           label=fr'$\phi_1^*$ MJD = {PHI1_MJD:.4f}')
ax.set_xlabel('Iteration')
ax.set_ylabel(r'$\phi_1$', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
savefig('figure1_convergence_real')


# ─────────────────────────────────────────────────────────────────────────────
# FIGURE convergenza_RL — Training convergence (real data only, for thesis)
# ─────────────────────────────────────────────────────────────────────────────
print("\n[Figure convergenza_RL] Training convergence (real data)...")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(iters[w-1:], smooth(h_real, w),
        color=COLORS['real'], lw=2.0, label='Real S&P 500 (smoothed)')
ax.axhline(PHI1_BS,  color=COLORS['bs'],  lw=1.2, ls='--',
           label=fr'$\phi_1^*$ BS = {PHI1_BS:.4f}')
ax.axhline(PHI1_MJD, color=COLORS['mjd'], lw=1.2, ls='-.',
           label=fr'$\phi_1^*$ MJD = {PHI1_MJD:.4f}')
ax.set_xlabel('Iteration')
ax.set_ylabel(r'$\phi_1$', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
zoom = N_ITER // 2
ax.plot(iters[zoom:],       h_real[zoom:],
        color=COLORS['real'], lw=0.5, alpha=0.3)
ax.plot(iters[zoom+w-1:],  smooth(h_real[zoom:], w),
        color=COLORS['real'], lw=2.0, label='Real data (smoothed)')
ax.axhline(PHI1_BS,  color=COLORS['bs'],  lw=1.5, ls='--',
           label=fr'$\phi_1^*$ BS = {PHI1_BS:.4f}')
ax.axhline(PHI1_MJD, color=COLORS['mjd'], lw=1.5, ls='-.',
           label=fr'$\phi_1^*$ MJD = {PHI1_MJD:.4f}')
ax.set_xlabel('Iteration')
ax.set_ylabel(r'$\phi_1$', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
savefig('convergenza_RL')


# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 2 — Terminal wealth distribution (bootstrap OOS)
# ─────────────────────────────────────────────────────────────────────────────
print("\n[Figure 2] Terminal wealth distribution...")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

all_tw = np.concatenate([tw_real, tw_bs, tw_mjd])
bins   = np.linspace(all_tw.min() - 0.05, all_tw.max() + 0.05, 60)

# Histogram / KDE
ax = axes[0]
entries = [
    (tw_real, fr'Real$\to$Real  ($\mu$={tw_real.mean():.3f}, $\sigma$={tw_real.std():.3f})',
     COLORS['real']),
    (tw_bs,   fr'BS$\to$Real    ($\mu$={tw_bs.mean():.3f}, $\sigma$={tw_bs.std():.3f})',
     COLORS['bs']),
    (tw_mjd,  fr'MJD$\to$Real   ($\mu$={tw_mjd.mean():.3f}, $\sigma$={tw_mjd.std():.3f})',
     COLORS['mjd']),
]
for tw, lbl, col in entries:
    ax.hist(tw, bins=bins, alpha=0.5, density=True, color=col, label=lbl)
ax.axvline(Z, color='black', lw=1.5, ls='--', label=f'Target $z={Z}$')
ax.set_xlabel(r'$X_T$', fontsize=12)
ax.set_ylabel('Density', fontsize=11)
ax.legend()
ax.grid(True, alpha=0.3)

# Boxplot
ax = axes[1]
bp = ax.boxplot([tw_real, tw_bs, tw_mjd],
                patch_artist=True, widths=0.5,
                medianprops=dict(color='black', lw=2))
for patch, col in zip(bp['boxes'],
                      [COLORS['real'], COLORS['bs'], COLORS['mjd']]):
    patch.set_facecolor(col)
    patch.set_alpha(0.6)
ax.axhline(Z, color='black', lw=1.5, ls='--', label=f'Target $z={Z}$')
ax.set_xticks([1, 2, 3])
ax.set_xticklabels([r'Real$\to$Real', r'BS$\to$Real', r'MJD$\to$Real'],
                   fontsize=10)
ax.set_ylabel(r'$X_T$', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
savefig('figure2_wealth_real')

# Print std values for thesis caption
print(f"  std values for caption: Real={tw_real.std():.3f}, "
      f"BS={tw_bs.std():.3f}, MJD={tw_mjd.std():.3f}")


# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 3 — Shape of the learned policy
# ─────────────────────────────────────────────────────────────────────────────
print("\n[Figure 3] Learned policy shape...")

T     = 1.0
wg    = np.linspace(0.5, 2.5, 200)

fig, ax = plt.subplots(figsize=(8, 5))

for phi, omega, lbl, col in [
    (phi_real, float(data['omega_real']), 'Real S&P 500 data', COLORS['real']),
    (phi_bs,   float(data['omega_bs']),   'BS simulator',      COLORS['bs']),
    (phi_mjd,  float(data['omega_mjd']),  'MJD simulator',     COLORS['mjd']),
]:
    actions = [-phi[0] * (x - omega) for x in wg]
    ax.plot(wg, actions, label=lbl, lw=2.0, color=col)

ax.axvline(Z, color='black', lw=1.0, ls=':', alpha=0.7,
           label=f'Target $z={Z}$')
ax.axhline(0, color='gray',  lw=0.8, ls='--', alpha=0.5)
ax.set_xlabel(r'Wealth $x$', fontsize=12)
ax.set_ylabel(r'Allocation $a_t$', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
savefig('figure3_policy_real')


# ─────────────────────────────────────────────────────────────────────────────
# FIGURE rl_out_of_sample — Chronological OOS wealth 2020-2023
# ─────────────────────────────────────────────────────────────────────────────
print("\n[Figure rl_out_of_sample] Chronological OOS 2020-2023...")

if oos_rl is None:
    print("  WARNING: OOS wealth paths not found in mv_results.npz.")
    print("  Re-run mv_RL.py to generate them.")
else:
    n_months = len(oos_rl) - 1   # 48
    months   = np.arange(n_months + 1)

    # Approximate month labels (Jan 2020 = 0)
    year_ticks  = [0, 12, 24, 36, 48]
    year_labels = ['Jan 2020', 'Jan 2021', 'Jan 2022', 'Jan 2023', 'Jan 2024']

    fig, ax = plt.subplots(figsize=(10, 5))

    ax.plot(months, oos_rl,  color=COLORS['real'], lw=2.0, ls='-',
            label=fr'RL policy ($\phi_1={phi_real[0]:.3f}$)')
    ax.plot(months, oos_bh,  color=COLORS['bh'],  lw=1.5, ls=':',
            label=r'Buy-and-hold ($a_t=0.4$)')
    ax.plot(months, oos_bs,  color=COLORS['bs'],  lw=1.5, ls='--',
            label=fr'Benchmark BS ($\phi_1^*={PHI1_BS:.3f}$)')
    ax.plot(months, oos_mjd, color=COLORS['mjd'], lw=1.5, ls='-.',
            label=fr'Benchmark MJD ($\phi_1^*={PHI1_MJD:.3f}$)')

    ax.axhline(Z, color='black', lw=1.0, ls=':', alpha=0.6,
               label=f'Target $z={Z}$')

    # Shade COVID crash and 2022 bear market
    ax.axvspan(1, 4,   alpha=0.08, color='red',  label='COVID crash')
    ax.axvspan(24, 36, alpha=0.08, color='blue', label='2022 bear market')

    ax.set_xticks(year_ticks)
    ax.set_xticklabels(year_labels, fontsize=9)
    ax.set_xlabel('Month', fontsize=12)
    ax.set_ylabel(r'Wealth $X_t$', fontsize=12)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    savefig('rl_out_of_sample')

    # Print final wealth values
    print(f"  Terminal wealth: RL={oos_rl[-1]:.3f}, B&H={oos_bh[-1]:.3f}, "
          f"BS={oos_bs[-1]:.3f}, MJD={oos_mjd[-1]:.3f}")

print("\nAll figures saved.")

Loading results from: mv_results.npz
  N_ITER = 20000
  Final phi_1:  Real=1.2242  BS=1.1979  MJD=1.4309
  Analytical phi_1*:  BS=1.2017  MJD=1.2810
  Output folder: .

[Figure 1] Convergence of phi_1...


C:\Users\debor\AppData\Local\Temp\ipykernel_26640\3094802886.py:70: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  PHI1_BS  = float(data['PHI1_BS'])
C:\Users\debor\AppData\Local\Temp\ipykernel_26640\3094802886.py:71: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  PHI1_MJD = float(data['PHI1_MJD'])


  Saved: .\figure1_convergence_real.pdf
  Saved: .\figure1_convergence_real.png

[Figure convergenza_RL] Training convergence (real data)...
  Saved: .\convergenza_RL.pdf
  Saved: .\convergenza_RL.png

[Figure 2] Terminal wealth distribution...
  Saved: .\figure2_wealth_real.pdf
  Saved: .\figure2_wealth_real.png
  std values for caption: Real=0.085, BS=0.087, MJD=0.100

[Figure 3] Learned policy shape...


C:\Users\debor\AppData\Local\Temp\ipykernel_26640\3094802886.py:254: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  (phi_real, float(data['omega_real']), 'Real S&P 500 data', COLORS['real']),
C:\Users\debor\AppData\Local\Temp\ipykernel_26640\3094802886.py:255: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  (phi_bs,   float(data['omega_bs']),   'BS simulator',      COLORS['bs']),
C:\Users\debor\AppData\Local\Temp\ipykernel_26640\3094802886.py:256: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (

  Saved: .\figure3_policy_real.pdf
  Saved: .\figure3_policy_real.png

[Figure rl_out_of_sample] Chronological OOS 2020-2023...
  Saved: .\rl_out_of_sample.pdf
  Saved: .\rl_out_of_sample.png
  Terminal wealth: RL=1.187, B&H=1.165, BS=1.184, MJD=1.194

All figures saved.
